# numcompute_stream Demo

**Assignment 2.2 | Programming for AI**

This notebook demonstrates an end-to-end streaming machine learning pipeline built entirely with **plain Python + NumPy + matplotlib** — no scikit-learn, no pandas.

### What this demo covers

| Step | Description |
|------|-------------|
| 1 | Load `dataset.csv` using the custom `io.py` pipeline |
| 2 | Split into chunks to simulate a streaming data source |
| 3 | Build three `Pipeline` instances (DecisionTree, RandomForest, AdaBoost) |
| 4 | Train each pipeline incrementally via `StreamTrainer.fit_chunk()` |
| 5 | Evaluate after every chunk via `StreamTrainer.score_chunk()` |
| 6 | Log and visualise metrics over time using `visualise.py` |
| 7 | Print the `StreamTrainer` summary table |
| 8 | Save training logs to CSV |
| 9 | Show per-chunk descriptive statistics via `update_stats()` |

## Setup

In [1]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Make the package importable when running from the demo/ folder
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from numcompute_stream import (
    Pipeline,
    StandardScaler,
    DecisionTreeClassifier,
    EnsembleClassifier,
    StreamTrainer,
    visualise,
)
from numcompute_stream.io import (
    read_csv,
    split_into_chunks,
    train_test_split,
)
from numcompute_stream.metrics import (
    Accuracy,
    confusion_matrix as batch_cm,
)
from numcompute_stream.stats import update_stats, reset_stats

# Config
N_CHUNKS    = 12
RANDOM_SEED = 42
DATA_PATH   = os.path.join("data", "dataset.csv")
OUT_DIR     = os.path.join("outputs")
os.makedirs(OUT_DIR, exist_ok=True)

print("All imports OK")

All imports OK


---
## Step 1 — Load the Dataset

We use `io.read_csv()` — a custom CSV parser built with NumPy — to load the customer churn dataset. No pandas.

In [2]:
headers, data = read_csv(DATA_PATH)

X = data[:, :-1]
y = data[:, -1].astype(int)
feature_names = headers[:-1]

classes, counts = np.unique(y, return_counts=True)

print(f"File       : {DATA_PATH}")
print(f"Samples    : {X.shape[0]}")
print(f"Features   : {X.shape[1]}  →  {feature_names}")
print(f"Classes    : churn=0 (stay) → {counts[0]} customers")
print(f"             churn=1 (left) → {counts[1]} customers")
print(f"Churn rate : {counts[1] / len(y) * 100:.1f}%")

File       : data\dataset.csv
Samples    : 1200
Features   : 9  →  ['age', 'tenure_months', 'monthly_charges', 'total_charges', 'num_products', 'support_calls', 'avg_call_duration', 'has_contract', 'satisfaction_score']
Classes    : churn=0 (stay) → 913 customers
             churn=1 (left) → 287 customers
Churn rate : 23.9%


---
## Step 2 — Train/Test Split and Chunk Creation

`split_into_chunks()` divides the training data into `N_CHUNKS` equal-sized batches, simulating data arriving in a stream. The test set is held out completely and never seen during training.

In [3]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

chunks = split_into_chunks(
    X_tr, y_tr, n_chunks=N_CHUNKS,
    shuffle=True, random_state=RANDOM_SEED
)

print(f"Train set  : {len(y_tr)} customers  →  {N_CHUNKS} chunks of ~{len(chunks[0][1])} each")
print(f"Test set   : {len(y_te)} customers (held out)")
print(f"Chunk sizes: {[len(yc) for _, yc in chunks]}")

Train set  : 960 customers  →  12 chunks of ~80 each
Test set   : 240 customers (held out)
Chunk sizes: [80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80]


---
## Step 3 — Build Three Streaming Pipelines

Each `Pipeline` chains a `StandardScaler` (using Welford's online algorithm for streaming updates) with a model. All three share the same interface — `partial_fit`, `predict`, `score` — so they can be trained and evaluated identically.

| Pipeline | Model | Key Setting |
|----------|-------|-------------|
| DecisionTree | `DecisionTreeClassifier` | `max_depth=6, criterion='gini'` |
| RandomForest | `EnsembleClassifier(method='random_forest')` | 8 trees, `sqrt(d)` features per split |
| AdaBoost | `EnsembleClassifier(method='adaboost')` | 8 boosting rounds |

In [4]:
pipelines = {
    "DecisionTree": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  DecisionTreeClassifier(max_depth=6, criterion="gini", random_state=0)),
    ]),
    "RandomForest": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  EnsembleClassifier(
            method="random_forest", n_estimators=8, max_depth=5, random_state=0
        )),
    ]),
    "AdaBoost": Pipeline([
        ("scaler", StandardScaler()),
        ("model",  EnsembleClassifier(
            method="adaboost", n_estimators=8, random_state=0
        )),
    ]),
}

print("Pipelines built:")
for name, pipe in pipelines.items():
    print(f"  {name}: {pipe}")

Pipelines built:
  DecisionTree: Pipeline([
  ('scaler', StandardScaler)
  ('model', DecisionTreeClassifier)
])
  RandomForest: Pipeline([
  ('scaler', StandardScaler)
  ('model', EnsembleClassifier)
])
  AdaBoost: Pipeline([
  ('scaler', StandardScaler)
  ('model', EnsembleClassifier)
])


---
## Step 4 — Incremental Training via StreamTrainer

`StreamTrainer` wraps any `Pipeline` and handles all bookkeeping automatically:
- wall-clock time per chunk
- memory footprint of each incoming chunk
- training accuracy on the chunk
- evaluation accuracy on the held-out test set
- cumulative accuracy across all chunks so far

Logging is kept **separate from model logic** — the pipeline stays clean and independently testable.

In [5]:
trainers = {
    name: StreamTrainer(pipeline=pipe, verbose=False)
    for name, pipe in pipelines.items()
}

header = f"{'Chunk':>5}  {'Customers':>10}  " + "  ".join(f"{n:>14}" for n in pipelines)
print(header)
print("─" * len(header))

for idx, (Xc, yc) in enumerate(chunks):
    row = f"{idx+1:>5}  {len(yc):>10}  "
    for name, trainer in trainers.items():
        trainer.fit_chunk(Xc, yc)
        acc = trainer.score_chunk(X_te, y_te)
        row += f"{acc:>14.4f}  "
    print(row)

Chunk   Customers    DecisionTree    RandomForest        AdaBoost
─────────────────────────────────────────────────────────────────
    1          80          0.5917          0.7167          0.7125  
    2          80          0.6667          0.7417          0.7250  
    3          80          0.6500          0.7250          0.7167  
    4          80          0.6750          0.7542          0.7292  
    5          80          0.7083          0.7458          0.7083  
    6          80          0.6875          0.7333          0.7375  
    7          80          0.7083          0.7292          0.7375  
    8          80          0.7250          0.7375          0.7333  
    9          80          0.6958          0.7417          0.7375  
   10          80          0.7042          0.7375          0.7458  
   11          80          0.7458          0.7417          0.7375  
   12          80          0.6833          0.7375          0.7375  


---
## Step 5 — Final Evaluation on the Held-Out Test Set

In [6]:
best_name = None
best_acc  = -1.0
final_accs = {}

for name, pipe in pipelines.items():
    acc    = pipe.score(X_te, y_te)
    preds  = pipe.predict(X_te)
    n_correct = int(np.sum(preds == y_te))
    final_accs[name] = acc
    print(f"{name:<14}: accuracy={acc:.4f}  ({n_correct}/{len(y_te)} correct)")
    if acc > best_acc:
        best_acc  = acc
        best_name = name

print(f"\nBest model: {best_name}  (acc={best_acc:.4f})")

DecisionTree  : accuracy=0.6833  (164/240 correct)
RandomForest  : accuracy=0.7375  (177/240 correct)
AdaBoost      : accuracy=0.7375  (177/240 correct)

Best model: RandomForest  (acc=0.7375)


---
## Step 6 — Visualisations

All plots use `visualise.py` — a custom matplotlib module. Each figure is displayed inline and also saved to `demo/outputs/`.

### 6a — Test Accuracy over Chunks (per model)

In [7]:
for name, trainer in trainers.items():
    history = trainer.get_metric_history("eval_acc")
    fig = visualise.plot_metric_over_time(
        history,
        title     = f"{name} — Test Accuracy per Chunk",
        ylabel    = "Accuracy",
        xlabel    = "Chunk",
        smoothing = 3,
        save_path = os.path.join(OUT_DIR, f"{name.lower()}_accuracy.png"),
    )
    plt.show()
    plt.close(fig)

C:\Users\Jaineel\AppData\Local\Temp\ipykernel_44892\3885435574.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6b — Model Comparison: DecisionTree vs RandomForest

In [8]:
names = list(trainers.keys())
fig = visualise.compare_models(
    trainers[names[0]].get_metric_history("eval_acc"),
    trainers[names[1]].get_metric_history("eval_acc"),
    labels    = [names[0], names[1]],
    title     = f"Model Comparison: {names[0]} vs {names[1]}",
    ylabel    = "Accuracy",
    save_path = os.path.join(OUT_DIR, "model_comparison.png"),
)
plt.show()
plt.close(fig)

C:\Users\Jaineel\AppData\Local\Temp\ipykernel_44892\1784098282.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6c — Predictions vs Ground Truth (Best Model)

In [9]:
best_pipe = pipelines[best_name]
y_pred    = best_pipe.predict(X_te)

fig = visualise.plot_predictions_vs_ground_truth(
    y_te,
    y_pred,
    title     = f"{best_name} — Churn Predictions vs Actual (Test Set)",
    save_path = os.path.join(OUT_DIR, "predictions_vs_truth.png"),
)
plt.show()
plt.close(fig)

C:\Users\Jaineel\AppData\Local\Temp\ipykernel_44892\3629155642.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6d — Confusion Matrix

In [10]:
mat, cls = batch_cm(y_te, y_pred)

fig = visualise.plot_confusion_matrix(
    mat,
    ["Stay (0)", "Churn (1)"],
    title     = f"{best_name} — Confusion Matrix",
    normalise = False,
    save_path = os.path.join(OUT_DIR, "confusion_matrix.png"),
)
plt.show()
plt.close(fig)

C:\Users\Jaineel\AppData\Local\Temp\ipykernel_44892\597514997.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6e — Learning Curve (Samples Seen vs Accuracy)

In [11]:
best_trainer  = trainers[best_name]
eval_history  = best_trainer.get_metric_history("eval_acc")
train_history = best_trainer.get_metric_history("train_acc")
n_samples_log = [
    entry["n_total"]
    for entry in best_trainer.log_
    if entry.get("eval_acc") is not None
]

fig = visualise.plot_learning_curve(
    n_samples_list = n_samples_log,
    train_scores   = train_history[:len(n_samples_log)],
    val_scores     = eval_history,
    metric_name    = "Accuracy",
    title          = f"{best_name} — Learning Curve (Customers Seen vs Accuracy)",
    save_path      = os.path.join(OUT_DIR, "learning_curve.png"),
)
plt.show()
plt.close(fig)

print(f"All plots saved to {OUT_DIR}/")

All plots saved to outputs/


C:\Users\Jaineel\AppData\Local\Temp\ipykernel_44892\1143668650.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Step 7 — StreamTrainer Summary Table

`summary()` prints a formatted per-chunk log showing chunk size, cumulative samples, train/eval accuracy, memory footprint, and wall-clock time. The growing time per chunk reflects the O(N) rebuild cost as the data buffer accumulates.

In [12]:
print(f"StreamTrainer summary — best model: {best_name}\n")
trainers[best_name].summary()

StreamTrainer summary — best model: RandomForest

────────────────────────────────────────────────────────────────────────
 Chunk       N    Total   TrainAcc   EvalAcc   CumAcc   Mem(KB)       ms
────────────────────────────────────────────────────────────────────────
     1      80       80     0.8625    0.7167   0.7167       6.2    662.4
     2      80      160     0.8625    0.7417   0.7292       6.2   1224.0
     3      80      240     0.8500    0.7250   0.7278       6.2   1684.7
     4      80      320     0.8625    0.7542   0.7344       6.2   3504.8
     5      80      400     0.8000    0.7458   0.7367       6.2   3692.9
     6      80      480     0.7125    0.7333   0.7361       6.2   3697.5
     7      80      560     0.8000    0.7292   0.7351       6.2   4324.7
     8      80      640     0.8500    0.7375   0.7354       6.2   4605.8
     9      80      720     0.8125    0.7417   0.7361       6.2   5742.1
    10      80      800     0.7750    0.7375   0.7362       6.2   7195.0
 

---
## Step 8 — Save Training Logs to CSV

`log_to_csv()` writes the full per-chunk log for each model to `demo/outputs/<model>_log.csv`.

In [13]:
for name, trainer in trainers.items():
    path = os.path.join(OUT_DIR, f"{name.lower()}_log.csv")
    trainer.log_to_csv(path)
    print(f"Saved: {path}")

Log saved to outputs\decisiontree_log.csv
Saved: outputs\decisiontree_log.csv
Log saved to outputs\randomforest_log.csv
Saved: outputs\randomforest_log.csv
Log saved to outputs\adaboost_log.csv
Saved: outputs\adaboost_log.csv


---
## Step 9 — Streaming Descriptive Statistics via `update_stats()`

`update_stats()` in `stats.py` maintains running mean, std, min, and max incrementally using Welford's algorithm. Here we track `monthly_charges` (column index 2) as it arrives chunk by chunk — no access to future data.

In [14]:
col_idx  = 2
col_name = feature_names[col_idx]

reset_stats()
print(f"Tracking column: '{col_name}'\n")
print(f"{'Chunk':>6}  {'Mean':>10}  {'Std':>10}  {'Min':>10}  {'Max':>10}  {'N':>6}")
print("─" * 58)

for idx, (Xc, _) in enumerate(chunks):
    col_data = Xc[:, col_idx].reshape(-1, 1)
    stats    = update_stats(col_data)

    mean_val = stats["mean"][0] if hasattr(stats["mean"], "__len__") else stats["mean"]
    std_val  = stats["std"][0]  if hasattr(stats["std"],  "__len__") else stats["std"]
    min_val  = stats["min"][0]  if hasattr(stats["min"],  "__len__") else stats["min"]
    max_val  = stats["max"][0]  if hasattr(stats["max"],  "__len__") else stats["max"]

    print(
        f"{idx+1:>6}  {mean_val:>10.2f}  {std_val:>10.2f}  "
        f"{min_val:>10.2f}  {max_val:>10.2f}  {stats['n']:>6}"
    )

Tracking column: 'monthly_charges'

 Chunk        Mean         Std         Min         Max       N
──────────────────────────────────────────────────────────
     1       62.87       28.40       20.32      117.28      80
     2       66.35       28.09       20.32      119.71     160
     3       66.50       28.40       20.19      119.71     240
     4       66.55       28.56       20.19      119.71     320
     5       67.20       28.46       20.19      119.71     400
     6       67.36       28.58       20.19      119.95     480
     7       68.40       28.53       20.19      119.95     560
     8       68.14       28.51       20.19      119.95     640
     9       68.58       28.59       20.02      119.95     720
    10       68.80       28.99       20.02      119.95     800
    11       69.19       29.04       20.02      119.97     880
    12       69.67       29.12       20.02      119.97     960


---
## Summary

This notebook demonstrated a complete streaming ML workflow using **numcompute_stream** — built with NumPy only.

In [15]:
print("══════════════════════════════════════════════════════════════")
print("  Demo Complete")
print("══════════════════════════════════════════════════════════════")
print(f"  Dataset      : Customer Churn ({len(y)} customers)")
print(f"  Features     : {feature_names}")
print(f"  Chunks       : {N_CHUNKS}")
print(f"  Train        : {len(y_tr)} customers")
print(f"  Test         : {len(y_te)} customers")
print(f"  Best model   : {best_name}  (acc={final_accs[best_name]:.4f})")
print(f"  Plots saved  : {OUT_DIR}/")

══════════════════════════════════════════════════════════════
  Demo Complete
══════════════════════════════════════════════════════════════
  Dataset      : Customer Churn (1200 customers)
  Features     : ['age', 'tenure_months', 'monthly_charges', 'total_charges', 'num_products', 'support_calls', 'avg_call_duration', 'has_contract', 'satisfaction_score']
  Chunks       : 12
  Train        : 960 customers
  Test         : 240 customers
  Best model   : RandomForest  (acc=0.7375)
  Plots saved  : outputs/
